### 1. Setup
We start by loading environment variables from a `.env` file and initializing the Gemini client using the `google-genai` SDK.

In [1]:
# Load env variables and create client
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client()
model_id = "gemini-2.5-flash-lite"

### 2. Helper Functions
These helper functions manage the conversation history and facilitate interactions with the Gemini model.

In [2]:
# Helper functions
from google.genai import types

def add_user_message(messages, message):
    messages.append({"role": "user", "parts": [{"text": message}]})

def add_assistant_message(messages, content):
    messages.append({"role": "model", "parts": content.parts})

def chat(messages, system=None, temperature=1.0, tools=None):
    config = types.GenerateContentConfig(
        system_instruction=system,
        temperature=temperature,
        tools=tools,
    )
    
    response = client.models.generate_content(
        model=model_id,
        contents=messages,
        config=config
    )
    return response


### 3. Google Search Tool
Gemini provides a built-in Google Search tool that allows the model to retrieve real-time information from the web.

In [3]:
# Define the Google Search tool
google_search_tool = types.Tool(
    google_search=types.GoogleSearch()
)
tools = [google_search_tool]

### 4. Executing Search
We send a prompt to Gemini with the Google Search tool enabled. Gemini will autonomously decide when to use the tool to provide an up-to-date answer.

In [4]:
messages = []
prompt = "What's the best exercise for gaining chest muscle (for men)?"
add_user_message(messages, prompt)

print(f"User: {prompt}\n")

response = chat(messages, tools=tools)

print("Assistant:")
for part in response.candidates[0].content.parts:
    if part.text:
        print(part.text)

# Show grounding metadata if available
if response.candidates[0].grounding_metadata:
    print("\nGrounding Sources:")
    for chunk in response.candidates[0].grounding_metadata.grounding_chunks:
        if chunk.web:
            print(f"- {chunk.web.title}: {chunk.web.uri}")

User: What's the best exercise for gaining chest muscle (for men)?

Assistant:
To gain chest muscle, men should focus on exercises that target the pectoral muscles from different angles and incorporate progressive overload. The most effective exercises for chest muscle growth include:

*   **Barbell Bench Press:** This is a foundational exercise for building overall chest mass and strength, particularly for the middle chest.
*   **Incline Dumbbell Press:** This targets the upper chest and is effective for developing a fuller, more balanced chest. Using dumbbells increases the range of motion and muscle activation.
*   **Weighted Dips:** Particularly when leaning slightly forward, dips are excellent for targeting the lower chest and building strength.
*   **Dumbbell Chest Fly:** This isolation exercise helps to activate more muscle fibers by creating a deep stretch in the chest.
*   **Push-Ups:** A highly accessible exercise that engages the pectoralis major and triceps. Variations like